# Optunaによるパラメータのオートチューニング

In [ ]:
!pip install -qq optuna kaggle-environments

In [ ]:
"""Colabで実行するパラメータ探索コード。"""

import importlib
import statistics

import optuna
from kaggle_environments import make

import main as strategy


# Colab上の最新のmain.pyを読み込む。
strategy = importlib.reload(strategy)

N_TRIALS = 100
MATCH_COUNT = 20
EVALUATION_SEEDS = list(range(MATCH_COUNT))


def objective(trial):
    """平均得点を返す。"""

    #いちごの種数と植付済み数を合わせた購入目標数
    strategy.StrategyConfig.STRAWBERRY_TARGET_COUNT = trial.suggest_int(
        "STRAWBERRY_TARGET_COUNT",
        10,
        50,
        step=2,
    )

    #いちごの植付けと種購入を許可する期限
    strategy.StrategyConfig.STRAWBERRY_PLANT_END_DAY = trial.suggest_int(
        "STRAWBERRY_PLANT_END_DAY",
        14,
        30,
        step=2
    )

    #いちご需要店舗が何店舗以上なら種を購入するか
    strategy.StrategyConfig.MIN_STRAWBERRY_DEMAND_SHOPS = trial.suggest_int(
        "MIN_STRAWBERRY_DEMAND_SHOPS",
        0,
        5,
    )



    rewards = []

    for episode_seed in EVALUATION_SEEDS:
        strategy.hire_controller = strategy.HireController()

        env = make(
            "kaggriculture",
            configuration={
                "episodeSteps": 720,
                "seed": episode_seed,
            },
            debug=True,
        )

        env.run([strategy.agent, strategy.agent])

        for state in env.steps[-1]:
            rewards.append(float(state.reward))

    return statistics.fmean(rewards)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=1,
    show_progress_bar=True,
)

print("\n===== 最高平均得点 =====")
print(study.best_value)

print("\n===== main.pyへ手動設定する値 =====")

for name, value in study.best_params.items():
    print(f"{name} = {value}")

[I 2026-09-09 11:00:46,249] A new study created in memory with name: no-name-851e059e-b688-4cde-96d1-0fc30a8585a1


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-09-09 11:02:26,872] Trial 0 finished with value: 46431.65 and parameters: {'STRAWBERRY_TARGET_COUNT': 24, 'STRAWBERRY_PLANT_END_DAY': 30, 'MIN_STRAWBERRY_DEMAND_SHOPS': 4}. Best is trial 0 with value: 46431.65.
[I 2026-09-09 11:04:19,414] Trial 1 finished with value: 65857.85 and parameters: {'STRAWBERRY_TARGET_COUNT': 34, 'STRAWBERRY_PLANT_END_DAY': 16, 'MIN_STRAWBERRY_DEMAND_SHOPS': 0}. Best is trial 1 with value: 65857.85.
[I 2026-09-09 11:05:57,118] Trial 2 finished with value: 46922.5 and parameters: {'STRAWBERRY_TARGET_COUNT': 12, 'STRAWBERRY_PLANT_END_DAY': 28, 'MIN_STRAWBERRY_DEMAND_SHOPS': 3}. Best is trial 1 with value: 65857.85.
[I 2026-09-09 11:07:30,073] Trial 3 finished with value: 41257.35 and parameters: {'STRAWBERRY_TARGET_COUNT': 38, 'STRAWBERRY_PLANT_END_DAY': 14, 'MIN_STRAWBERRY_DEMAND_SHOPS': 5}. Best is trial 1 with value: 65857.85.
[I 2026-09-09 11:09:23,695] Trial 4 finished with value: 68626.65 and parameters: {'STRAWBERRY_TARGET_COUNT': 44, 'STRAWBERRY